## Config

In [1]:
import pandas as pd
import numpy as np
import re
import time
import os
from dotenv import load_dotenv
from tqdm.auto import tqdm


from huggingface_hub import login

import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM


In [2]:
raw_path = os.path.join("data", "raw", 'projetos-lattes.csv')
dt = pd.read_csv(raw_path, sep=';', encoding='utf-8-sig')
dt.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
0,2847901878340953,Antonio Lopes Apolinario Junior,2016,2024,Instituto Nacional de Ciência e Tecnologia em ...,Descrição: Realizar pesquisa e desenvolvimento...
1,2847901878340953,Antonio Lopes Apolinario Junior,2009,2014,REALIDADE AUMENTADA APLICADA A BIOENGENHARIA C...,Descrição: Este projeto tem como foco a criaçã...
2,2847901878340953,Antonio Lopes Apolinario Junior,2008,2014,INCT-MACC Instituto Nacional de Ciencia e Tecn...,Descrição: Com a materialização do INCT MACC t...
3,2847901878340953,Antonio Lopes Apolinario Junior,2008,2013,Rede de Cooperação para Análise de Dados e Mod...,Descrição: Neste projeto pretendemos criar uma...
4,2847901878340953,Antonio Lopes Apolinario Junior,2008,2011,Construção de uma Colônia de Robôs Autônomos p...,Descrição: O projeto visa desenvolver uma colô...


In [3]:
dt["descricao"][0]

'Descrição: Realizar pesquisa e desenvolvimento em computação científica e suas aplicações na medicina, em especial a modelagem e simulação computacional dos sistemas fisiológicos que integram o corpo humano; promover o desenvolvimento do processamento de imagens médicas, da visualização científica e da realidade virtual no desenvolvimento de aplicativos médicos orientados para a diagnose auxiliada por computador, tratamento, planejamento cirúrgico, treinamento e credenciamento médico; empregando as mais modernas técnicas de comunicação e transmissão multimídia desenvolver e gerenciar ambientes computacionais de alto desempenho que atendam às necessidades da medicina assistida por computação científica do País; formar recursos humanos e promover transferência de tecnologia e inovação para a área da saúde assistida por computação científica.. Situação: Concluído; Natureza: Pesquisa. Alunos envolvidos: Mestrado acadêmico: (1) Doutorado: (2) . Integrantes: Antonio Lopes Apolinario Junior 

In [4]:
t = dt.loc[dt["projeto"].str.contains("Edital")]
t["projeto"].values[5]

'Mapa Sonoro do Atlas Linguístico do Brasil PROINTER Edital 023/2014)'

In [5]:
dt.info()

<class 'pandas.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   lattes_id      368 non-null    int64
 1   professor      368 non-null    str  
 2   ano_inicio     368 non-null    int64
 3   ano_conclusao  368 non-null    str  
 4   projeto        368 non-null    str  
 5   descricao      368 non-null    str  
dtypes: int64(2), str(4)
memory usage: 17.4 KB


In [6]:
dt.shape

(368, 6)

In [7]:
dt.isna().sum()

lattes_id        0
professor        0
ano_inicio       0
ano_conclusao    0
projeto          0
descricao        0
dtype: int64

In [8]:
df = dt.copy(deep=True)

## Filtering to only projects run in 2025 and 2026

In [9]:
df["ano_conclusao"].unique()

<StringArray>
[ '2024',  '2014',  '2013',  '2011',  '2010',  '2009',  '2025', 'Atual',
  '2023',  '2022',  '2021',  '2019',  '2018',  '2017',  '2012',  '2026',
  '2016',  '2015',  '2007',  '2004',  '2003',  '2008',  '2020',  '2005',
  '2006',  '2002']
Length: 26, dtype: str

In [10]:
df["ano_conclusao"] = np.where(df["ano_conclusao"] == "Atual", np.nan, df["ano_conclusao"])
df["ano_conclusao"] = pd.to_numeric(df["ano_conclusao"])
df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
0,2847901878340953,Antonio Lopes Apolinario Junior,2016,2024.0,Instituto Nacional de Ciência e Tecnologia em ...,Descrição: Realizar pesquisa e desenvolvimento...
1,2847901878340953,Antonio Lopes Apolinario Junior,2009,2014.0,REALIDADE AUMENTADA APLICADA A BIOENGENHARIA C...,Descrição: Este projeto tem como foco a criaçã...
2,2847901878340953,Antonio Lopes Apolinario Junior,2008,2014.0,INCT-MACC Instituto Nacional de Ciencia e Tecn...,Descrição: Com a materialização do INCT MACC t...
3,2847901878340953,Antonio Lopes Apolinario Junior,2008,2013.0,Rede de Cooperação para Análise de Dados e Mod...,Descrição: Neste projeto pretendemos criar uma...
4,2847901878340953,Antonio Lopes Apolinario Junior,2008,2011.0,Construção de uma Colônia de Robôs Autônomos p...,Descrição: O projeto visa desenvolver uma colô...


In [11]:
current = df.loc[df["ano_conclusao"].isna()]
print(f"{current.shape}\n")
current.head()

(75, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
9,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...
10,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...
12,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...
13,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...
32,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,NaN,CPA/IA - Uma Plataforma Digital Aberta Multius...,Descrição: O objetivo desta proposta é apresen...


### 2026

In [12]:
df["ano_inicio"].unique()

array([2016, 2009, 2008, 2006, 2024, 2023, 2022, 2021, 2020, 2018, 2017,
       2014, 2012, 2011, 2010, 2013, 2004, 2003, 2001, 2005, 2019, 2015,
       2007, 2002, 2000])

In [13]:
on_going_df = df.loc[df["ano_inicio"] == 2026]
print(f"{on_going_df.shape}\n")
on_going_df.head()

(0, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao


In [14]:
finished_this_year = df.loc[df["ano_conclusao"] == 2026]
print(f"{finished_this_year.shape}\n")
finished_this_year.head()

(11, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
28,5075736089100544,Cássio Vinicius Serafim Prazeres,2024,2026.0,Argus,"Descrição: No cenário atual, a segurança patri..."
29,5075736089100544,Cássio Vinicius Serafim Prazeres,2024,2026.0,Aprendizado para IoT e Dispositivos Federados ...,Descrição: Desenvolver um arcabouço para a exe...
30,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,2026.0,Uma Plataforma Inteligente para Redução de Que...,Descrição: A queda de pacientes em ambientes h...
38,5075736089100544,Cássio Vinicius Serafim Prazeres,2016,2026.0,SOFT-IoT: Self-Organizing Fog of Things,Situação: Concluído; Natureza: Pesquisa. Aluno...
133,6271096128174325,Frederico Araujo Durão,2023,2026.0,"SI2 - Instituto de Ciência, Tecnologia e Inova...","Descrição: Instituto de Ciência, Inovação e Te..."


In [15]:
this_year_df = pd.concat([
    current,
    on_going_df,
    finished_this_year
])
print(f"{this_year_df.shape}\n")
this_year_df.head()

(86, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
9,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...
10,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...
12,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...
13,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...
32,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,NaN,CPA/IA - Uma Plataforma Digital Aberta Multius...,Descrição: O objetivo desta proposta é apresen...


In [16]:
this_year_df["ano_conclusao"].unique()

array([  nan, 2026.])

In [17]:
this_year_df["ano_inicio"].unique()

array([2024, 2023, 2016, 2014, 2022, 2009, 2003, 2018, 2017, 2015, 2013,
       2021, 2020])

### 2025

In [18]:
# concluidos em 2025
fb = df.loc[df["ano_conclusao"] == 2025]
print(f"{fb.shape}\n")
fb.head()

(16, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
7,92226104911153,Bruno Pereira dos Santos,2024,2025.0,Explorando a Mobilidade em Redes Sem Fio com M...,Descrição: O projeto propõe o desenvolvimento ...
8,92226104911153,Bruno Pereira dos Santos,2024,2025.0,Explorando Relações Sociais na Internet das Co...,Descrição: O projeto proposto busca explorar o...
34,5075736089100544,Cássio Vinicius Serafim Prazeres,2022,2025.0,Orquestração de dados na Computação Urbana por...,Descrição: O objetivo deste projeto é aumentar...
80,9217378047217370,Daniela Barreiro Claro,2015,2025.0,CLOUD SECURITY INTEROPERABLE SOCIETY (CSIS): U...,Descrição: Este projeto visa desenvolver uma a...
91,8538761803577291,Eduardo Santana de Almeida,2021,2025.0,Uma Investigação sobre a Manutenibilidade de S...,Descrição: Este projeto de pesquisa tem três o...


In [19]:
# em andamento em 2025
og = current.loc[current["ano_inicio"] < 2026]
print(f"{og.shape}\n")
og.head()

(75, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
9,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...
10,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...
12,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...
13,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...
32,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,NaN,CPA/IA - Uma Plataforma Digital Aberta Multius...,Descrição: O objetivo desta proposta é apresen...


In [20]:
last_year_df = pd.concat([
    finished_this_year,
    fb,
    og 
])
print(f"{last_year_df.shape}\n")
last_year_df.head(3)

(102, 6)



,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao
28,5075736089100544,Cássio Vinicius Serafim Prazeres,2024,2026.0,Argus,"Descrição: No cenário atual, a segurança patri..."
29,5075736089100544,Cássio Vinicius Serafim Prazeres,2024,2026.0,Aprendizado para IoT e Dispositivos Federados ...,Descrição: Desenvolver um arcabouço para a exe...
30,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,2026.0,Uma Plataforma Inteligente para Redução de Que...,Descrição: A queda de pacientes em ambientes h...


In [21]:
last_year_df["ano_inicio"].unique()

array([2024, 2023, 2016, 2022, 2020, 2015, 2021, 2019, 2014, 2009, 2003,
       2018, 2017, 2013])

In [22]:
last_year_df["ano_conclusao"].unique()

array([2026., 2025.,   nan])

In [23]:
# both
both_df = pd.concat([
    this_year_df,
    last_year_df
]).drop_duplicates().reset_index(drop=True)
print(f"{both_df.shape}\n")
# both_df.to_csv(os.path.join("data", "output", "projetos_2025_2026.csv"), sep=";", encoding='utf-8-sig', index=False)

(102, 6)



# Raw Data Extraction

### Extraction hard coded

lattes_id: ID Lattes do professor.

professor: Nome do docente.

titulo_projeto: Nome completo do projeto.

ano_inicio: Ano de início.

ano_conclusao: Ano de término (ou "Atual").

situacao: Em andamento / Concluído.

funcao: Coordenador / Integrante.

financiadores_lattes: Lista de agências/empresas financiadoras declaradas.

natureza: Pesquisa / Desenvolvimento / Extensão.

descricao_completa: Texto descritivo bruto do Lattes.

In [24]:
def extrair_metadados_desc(texto):
    if not isinstance(texto, str):
        return pd.Series([None]*6)
    
    situacao = re.search(r'Situação:\s*([^;.]+)', texto)
    natureza = re.search(r'Natureza:\s*([^;.]+)', texto)
    alunos = re.search(r'Alunos envolvidos:\s*(.*?)(?=\.\s*Integrantes:|\.\s*Financiador|\Z)', texto)
    integrantes = re.search(r'Integrantes:\s*(.*?)(?=\.\s*Financiador|\Z)', texto)
    financiadores = re.search(r'Financiador\(es\):\s*(.+)', texto)
    desc_pura = re.search(r'Descrição:\s*(.*?)(?=\.\s*Situação:|\Z)', texto)
    
    return pd.Series([
        situacao.group(1).strip() if situacao else None,
        natureza.group(1).strip() if natureza else None,
        alunos.group(1).strip() if alunos else None,
        integrantes.group(1).strip() if integrantes else None,
        financiadores.group(1).strip() if financiadores else None,
        desc_pura.group(1).strip() if desc_pura else texto
    ])

In [25]:
def extrair_metadados_projeto(text):
    if not isinstance(text, str):
        return pd.Series([None]*1)
    
    edital = re.search(r'\((Edital.*?)\)', text)

    return pd.Series([
        edital.group(1).split("Edital ")[1] if edital else None
    ])

In [26]:
both_df[['situacao', 'natureza', 'alunos_envolvidos', 'integrantes', 'financiadores', 'descricao_limpa']] = both_df['descricao'].apply(extrair_metadados_desc)
both_df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa
0,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...,Em andamento,Pesquisa,Graduação: (4) / Mestrado acadêmico: (2) / Dou...,Bruno Pereira dos Santos - Integrante / Paulo ...,Conselho Nacional de Desenvolvimento Científic...,Situação: Em andamento; Natureza: Pesquisa. Al...
1,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Integrante / Maycon...,NaN,O aumento exponencial no volume de dados produ...
2,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...,Em andamento,Pesquisa,Mestrado acadêmico: (2) Doutorado: (2),Bruno Pereira dos Santos - Integrante / Heitor...,Fundação de Amparo à Pesquisa do Estado de Min...,Os ecossistemas de Internet das Coisas (IoT) o...
3,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Coordenador / Robes...,NaN,Situação: Em andamento; Natureza: Pesquisa. In...
4,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,NaN,CPA/IA - Uma Plataforma Digital Aberta Multius...,Descrição: O objetivo desta proposta é apresen...,Em andamento,Pesquisa,NaN,Cássio Vinicius Serafim Prazeres - Integrante ...,NaN,O objetivo desta proposta é apresentar a estra...


In [27]:
both_df[["edital"]] = both_df['projeto'].apply(extrair_metadados_projeto)
both_df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa,edital
0,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...,Em andamento,Pesquisa,Graduação: (4) / Mestrado acadêmico: (2) / Dou...,Bruno Pereira dos Santos - Integrante / Paulo ...,Conselho Nacional de Desenvolvimento Científic...,Situação: Em andamento; Natureza: Pesquisa. Al...,NaN
1,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Integrante / Maycon...,NaN,O aumento exponencial no volume de dados produ...,NaN
2,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...,Em andamento,Pesquisa,Mestrado acadêmico: (2) Doutorado: (2),Bruno Pereira dos Santos - Integrante / Heitor...,Fundação de Amparo à Pesquisa do Estado de Min...,Os ecossistemas de Internet das Coisas (IoT) o...,NaN
3,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Coordenador / Robes...,NaN,Situação: Em andamento; Natureza: Pesquisa. In...,NaN
4,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,NaN,CPA/IA - Uma Plataforma Digital Aberta Multius...,Descrição: O objetivo desta proposta é apresen...,Em andamento,Pesquisa,NaN,Cássio Vinicius Serafim Prazeres - Integrante ...,NaN,O objetivo desta proposta é apresentar a estra...,NaN


In [28]:
both_df["edital"].unique()

<StringArray>
[                                                nan,
                      'N 005/2022 - R$ 1.635.884,00',
            'Nº 005/2022 - INCITE - R$ 1.635.884,00',
  'CNPq/SEMPI/MCTI/FNDCT Nº 54/2022 - R$ 711.000,00',
              'CNPq| INCT 15/2008 - R$ 2.318.449,54',
                                           'CNPq - ',
 'CNPq 09/2023 - Bolsa de Produtividade em Pesquisa',
                                    'CNPq - PQ 2023',
             'N 005/2022 - INCITE - R$ 1.635.884,00',
   'CNPq/SEMPI/MCTI/FNDCT N 54/2022 - R$ 711.000,00',
  'CNPq - Bolsa Produtividade em Pesquisa N 09/2020',
       'Produtividade em Pesquisa PQ - CNPq 09/2021',
                              'SWE - CNPq N 26/2021']
Length: 13, dtype: str

In [29]:
both_df["ano_conclusao"].unique()

array([  nan, 2026., 2025.])

In [30]:
both_df["situacao"].unique()

<StringArray>
['Em andamento', 'Concluído']
Length: 2, dtype: str

#### Contagem alunos

In [31]:
def extrair_contagem_alunos(texto):
    if not isinstance(texto, str):
        return pd.Series([0, 0, 0, 0])
    
    grad = re.search(r'Graduação:\s*\(([0-9]+)\)', texto)
    mest_ac = re.search(r'Mestrado acadêmico:\s*\(([0-9]+)\)', texto)
    mest_prof = re.search(r'Mestrado profissional:\s*\(([0-9]+)\)', texto)
    dout = re.search(r'Doutorado:\s*\(([0-9]+)\)', texto)
    
    n_grad = int(grad.group(1)) if grad else 0
    n_mest = (int(mest_ac.group(1)) if mest_ac else 0) + (int(mest_prof.group(1)) if mest_prof else 0)
    n_dout = int(dout.group(1)) if dout else 0
    n_total = n_grad + n_mest + n_dout
    
    return pd.Series([n_grad, n_mest, n_dout, n_total])

In [32]:
both_df[['qtd_graduacao', 'qtd_mestrado', 'qtd_doutorado', 'total_alunos']] = both_df['descricao'].apply(extrair_contagem_alunos)

In [33]:
both_df.head()

,lattes_id,professor,ano_inicio,ano_conclusao,projeto,descricao,situacao,natureza,alunos_envolvidos,integrantes,financiadores,descricao_limpa,edital,qtd_graduacao,qtd_mestrado,qtd_doutorado,total_alunos
0,92226104911153,Bruno Pereira dos Santos,2024,NaN,Sistemas Resilientes para Ambientes Inteligent...,Situação: Em andamento; Natureza: Pesquisa. Al...,Em andamento,Pesquisa,Graduação: (4) / Mestrado acadêmico: (2) / Dou...,Bruno Pereira dos Santos - Integrante / Paulo ...,Conselho Nacional de Desenvolvimento Científic...,Situação: Em andamento; Natureza: Pesquisa. Al...,NaN,4,2,2,8
1,92226104911153,Bruno Pereira dos Santos,2024,NaN,Projeto ARGUS,Descrição: O aumento exponencial no volume de ...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Integrante / Maycon...,NaN,O aumento exponencial no volume de dados produ...,NaN,0,0,0,0
2,92226104911153,Bruno Pereira dos Santos,2023,NaN,Aprendizado de Máquina Federado para Internet ...,Descrição: Os ecossistemas de Internet das Coi...,Em andamento,Pesquisa,Mestrado acadêmico: (2) Doutorado: (2),Bruno Pereira dos Santos - Integrante / Heitor...,Fundação de Amparo à Pesquisa do Estado de Min...,Os ecossistemas de Internet das Coisas (IoT) o...,NaN,0,2,2,4
3,92226104911153,Bruno Pereira dos Santos,2023,NaN,Big Data Geocode: Uma aplicação de inteligênci...,Situação: Em andamento; Natureza: Pesquisa. In...,Em andamento,Pesquisa,NaN,Bruno Pereira dos Santos - Coordenador / Robes...,NaN,Situação: Em andamento; Natureza: Pesquisa. In...,NaN,0,0,0,0
4,5075736089100544,Cássio Vinicius Serafim Prazeres,2023,NaN,CPA/IA - Uma Plataforma Digital Aberta Multius...,Descrição: O objetivo desta proposta é apresen...,Em andamento,Pesquisa,NaN,Cássio Vinicius Serafim Prazeres - Integrante ...,NaN,O objetivo desta proposta é apresentar a estra...,NaN,0,0,0,0


### LLM feature extraction

area_pgcomp: Subárea principal de pesquisa no PGCOMP.

edital_ou_chamada: Nome do edital/chamada identificado no texto.

tem_fomento: Booleano (Sim/Não) ou agência identificada no texto.

palavras_chave_llm: Lista de palavras-chave padronizadas.

parceria_internacional: Booleano (Sim/Não) e país/instituição parceira.

resumo_executivo: Síntese concisa do objetivo.

bolsa: tipo da bolsa (Produtividade, Tecnologica, Outros)

In [34]:
# huggingface token
load_dotenv()
# print("token: ", os.getenv("HF_TOKEN"))

True

In [35]:
# confirm if the token belogs to the user
# from huggingface_hub import HfApi

# user_info = HfApi().whoami(token=os.getenv("HF_TOKEN"))
# print(f"Token ativo para a conta: {user_info['name']}")

In [36]:
model_id = "google/gemma-2-2b-it"
# model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cpu",
    local_files_only=True
)
model

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_lay

In [37]:
system = """
<role>
Você é um assistente acadêmico especialista em mineração de dados da Plataforma Lattes e análise de projetos de pesquisa do programa de mestrado de Ciência da Computação da UFBA.
</role>

<instrucoes>
1. Analise o texto do projeto fornecido e extraia ou deduza os campos definidos no schema JSON.
2. Idioma: Retorne todas as respostas em Português do Brasil.
3. Ausência de Informação: Se uma informação não estiver presente e não puder ser inferida com segurança, atribua o valor null ao campo (nunca invente dados).
4. Formato de Saída: Responda APENAS com o objeto JSON válido, sem texto explicativo antes ou depois, e sem marcações de código markdown (como ```json).
</instrucoes>

<taxonomia_pgcomp>
Para o campo 'area_pgcomp', escolha APENAS UMA das seguintes áreas do PGCOMP/UFBA que melhor descreve o projeto:
- Engenharia de Software
- Inteligência Artificial e Aprendizado de Máquina
- Redes de Computadores e Sistemas Distribuídos
- Ciência de Dados e Banco de Dados
- Segurança da Informação e Criptografia
- Visualização de Dados e Processamento de Imagens
- Computação Aplicada e Sistemas Interativos
- Otimização e Teoria da Computação
- Outra
</taxonomia_pgcomp>


<schema_json>
{
  "edital_ou_chamada": "string | null (Nome do edital ou chamada pública citada no texto, ex: 'Edital Universal CNPq 09/2023')",
  "coordenador": "string | null (Nome do coordenador do projeto. Extraia da lista de integrantes se houver a marcação 'Coordenador')",
  "valor_financiamento": "string | null (valor identificado no próprio texto caso apareça)",
  "financiador": "array | null (Lista com os nomes das agências de fomento, bolsas ou instituições financiadoras, ex: ['CNPq', 'CAPES'])",
  "tipo_bolsa": "Produtividade | Formação Acadêmica | Desenvolvimento Tecnológico | Apoio Técnico | Fixação Regional | null",
  "area_pgcomp": "string (Apenas uma área contida na <taxonomia_pgcomp>)",
  "topico_pesquisa": "string (Tema/tópico específico do projeto, ex: 'Testes e Qualidade de Software')",
  "palavras_chave": ["array com 3 a 5 palavras-chave técnicas em Português"],
  "cooperacao_internacional": "boolean (true se houver colaboração com universidades/instituições estrangeiras, senão false)",
  "resumo_executivo": "string (Resumo conciso de 1 a 2 frases com o objetivo principal do projeto)"
}
</schema_json>

<exemplo>
Entrada do Usuário:
"Descrição: O projeto visa investigar test smells no contexto dos testes manuais e automatizados de software para propor técnicas e estratégias. (Chamada CNPq 14/2023 Apoio a Projetos Internacionais). Integrantes: Fulano de Tal - Coordenador / Beltrano - Integrante. Financiador(es): CNPq - Auxílio financeiro."

Saída Esperada:
{
  "edital_ou_chamada": "CNPq 14/2023 Apoio a Projetos Internacionais",
  "coordenador": "Fulano de Tal",
  "financiador": ["CNPq"],
  "valor_financiamento": "R$ 2.400,00",
  "tipo_bolsa": "Produtividade",
  "area_pgcomp": "Engenharia de Software",
  "topico_pesquisa": "Teste de Software",
  "palavras_chave": ["Test Smells", "Testes de Software", "Qualidade de Software"],
  "cooperacao_internacional": true,
  "resumo_executivo": "O projeto investiga a ocorrência de test smells em testes manuais e automatizados de software para propor métodos e estratégias para diminuição do seu impacto na qualidade."
}
</exemplo>
"""

In [38]:
def processar_texto_llm(texto: str) -> dict:
    prompts = []
    messages = [
        {"role": "user", "content": f"{system}\n\nProjeto:\n{texto}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(prompt)
    print("messages: ", messages)
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cpu")
    
    output = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )
    # print("output: ", output)
    # print(type(output))
    input_len = inputs.input_ids.shape[1]

    resposta_bruta = tokenizer.decode(
        output[0][input_len:], skip_special_tokens=True
    ).strip()

    # print(f"resposta bruta:\n{resposta_bruta}\n")
    try:
        match = re.search(r"\{.*\}", resposta_bruta, re.DOTALL)
        if match:
            dados = json.loads(match.group(0))
        else:
            dados = {}
    except Exception as e:
        print(f"Erro ao decodificar JSON: {e}")
        dados = {}

    return dados

In [39]:
data_to_extract = both_df.copy(deep=True)
data_to_extract.info()

<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   lattes_id          102 non-null    int64  
 1   professor          102 non-null    str    
 2   ano_inicio         102 non-null    int64  
 3   ano_conclusao      27 non-null     float64
 4   projeto            102 non-null    str    
 5   descricao          102 non-null    str    
 6   situacao           102 non-null    str    
 7   natureza           102 non-null    str    
 8   alunos_envolvidos  65 non-null     str    
 9   integrantes        102 non-null    str    
 10  financiadores      53 non-null     str    
 11  descricao_limpa    102 non-null    str    
 12  edital             16 non-null     str    
 13  qtd_graduacao      102 non-null    int64  
 14  qtd_mestrado       102 non-null    int64  
 15  qtd_doutorado      102 non-null    int64  
 16  total_alunos       102 non-null    in

In [ ]:
data_to_extract = data_to_extract
textos_limpos = (
    data_to_extract["projeto"].astype(str)
    + " "
    + data_to_extract["descricao"].astype(str)
).tolist()
print(textos_limpos)
resultados_llm = []
arquivo_checkpoint = "checkpoint_resultados.jsonl"

print(f"Processando {len(textos_limpos)} projetos...")

loop_progress = tqdm(range(len(textos_limpos)), desc="Processando Projetos")

for i in loop_progress:
    lote = textos_limpos[i]
    
    init = time.time()
    res_lote = processar_texto_llm(lote)
    # print(res_lote)
    resultados_llm.extend(res_lote)
    fim = time.time()
    
    tempo = fim - init
    
    # Atualiza a barra com o tempo do último lote, em vez de usar print()
    loop_progress.set_postfix(tempo_lote=f"{tempo:.1f}s")
    
    # Gravação incremental
    with open(arquivo_checkpoint, "a", encoding="utf-8") as f:
        for res in res_lote:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

# --- ETAPA 4: MERGE FINAL ---
data_to_save = data_to_extract.copy(deep=True)
df_llm = pd.DataFrame(resultados_llm)
df_final = pd.concat([data_to_save, df_llm], axis=1)

df_final.head()

['Sistemas Resilientes para Ambientes Inteligentes: Abordando a Qualidade dos Dados e a Consciência de Contexto Situação: Em andamento; Natureza: Pesquisa. Alunos envolvidos: Graduação: (4) / Mestrado acadêmico: (2) / Doutorado: (2) . Integrantes: Bruno Pereira dos Santos - Integrante / Paulo H. L. Rettore - Coordenador. Financiador(es): Conselho Nacional de Desenvolvimento Científico e Tecnológico - Auxílio financeiro.', "Projeto ARGUS Descrição: O aumento exponencial no volume de dados produzidos pelos veículos autônomos conectados apresenta desafios em termos de comunicação, processamento e tomada de decisão na nuvem. Dessa forma, este projeto proposto visa explorar a aplicação da computação quântica na borda da rede para análise de dados de veículos autônomos conectados (CAVs), nomeada neste projeto de Borda Quântica. A hipótese adotada neste projeto é que a utilização para Borda Quântica para os veículos autônomos conectados possibilita a resolução de problemas complexos de anális

Processando Projetos:   0%|          | 0/102 [00:00<?, ?it/s]

messages:  [{'role': 'user', 'content': '\n<role>\nVocê é um assistente acadêmico especialista em mineração de dados da Plataforma Lattes e análise de projetos de pesquisa do programa de mestrado de Ciência da Computação da UFBA.\n</role>\n\n<instrucoes>\n1. Analise o texto do projeto fornecido e extraia ou deduza os campos definidos no schema JSON.\n2. Idioma: Retorne todas as respostas em Português do Brasil.\n3. Ausência de Informação: Se uma informação não estiver presente e não puder ser inferida com segurança, atribua o valor null ao campo (nunca invente dados).\n4. Formato de Saída: Responda APENAS com o objeto JSON válido, sem texto explicativo antes ou depois, e sem marcações de código markdown (como ```json).\n</instrucoes>\n\n<taxonomia_pgcomp>\nPara o campo \'area_pgcomp\', escolha APENAS UMA das seguintes áreas do PGCOMP/UFBA que melhor descreve o projeto:\n- Engenharia de Software\n- Inteligência Artificial e Aprendizado de Máquina\n- Redes de Computadores e Sistemas Dist

In [ ]:
print(df_final.shape)
print(df_final.columns.tolist())
save_path = os.path.join("data", "output")
if not os.path.exists(save_path):
    os.mkdir(save_path)
df_final.to_csv(os.path.join(save_path, 'projetos_lattes_completo.csv'), sep=';', index=False, encoding='utf-8-sig')
print("Processamento concluído com sucesso!")